In [2]:
import os
import sqlite3
import json
import numpy as np
import pandas as pd

current_dir = os.getcwd()
base_dir = os.path.abspath(os.path.join(current_dir, ".."))
RAW_DIR = os.path.join(base_dir, "data", "raw")
CLEAN_DIR = os.path.join(base_dir, "data", "clean")
TAB_DIR = os.path.join(base_dir, "outputs", "tables")

os.makedirs(CLEAN_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)

DISC_BINS = [-0.001, 0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.81]
DISC_LABELS = ["0%", "(0,10%]", "(10,20%]", "(20,30%]", "(30,40%]", "(40,50%]", ">50%"]

def drop_duplicates(df, log):
    n0 = len(df)
    dup_rowid = df["Row ID"].duplicated().sum()
    df = df.drop_duplicates(subset=["Row ID"], keep="first")
    
    key = ["Order ID", "Product ID", "Quantity", "Sales"]
    dup_biz = df.duplicated(subset=key).sum()
    df = df.drop_duplicates(subset=key, keep="first")
    n2 = len(df)

    log.append({"步骤": "重复值处理", "处理前": n0, "处理后": n2,
                "说明": f"Row ID 重复 {dup_rowid} 条；业务键重复 {dup_biz} 条，均保留首条"})
    return df

def handle_missing(df, log):
    n_miss = int(df.isna().sum().sum())
    miss_cols = df.columns[df.isna().any()].tolist()
    state_mode = df.groupby("State")["Postal Code"].agg(lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan)
    mask = df["Postal Code"].isna()
    df.loc[mask, "Postal Code"] = df.loc[mask, "State"].map(state_mode)
    df["Postal Code"] = df["Postal Code"].fillna(-1).astype("int64")
    log.append({"步骤": "缺失值处理", "处理前": n_miss, "处理后": int(df.isna().sum().sum()),
                "说明": f"缺失字段 {miss_cols}；按众数回填后被填 {int(mask.sum())} 条"})
    return df

def handle_outliers(df, log):
    n0 = len(df)
    bad_sales = df["Sales"] <= 0
    bad_qty = df["Quantity"] <= 0
    bad_date = df["Ship Date"] < df["Order Date"]
    df = df[~(bad_sales | bad_qty | bad_date)].copy()
    n1 = len(df)

    df["is_outlier"] = False
    for col in ["Sales", "Profit"]:
        q1, q3 = df[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        lo, hi = q1 - 3 * iqr, q3 + 3 * iqr
        df["is_outlier"] |= (df[col] < lo) | (df[col] > hi)

    log.append({"步骤": "异常值处理", "处理前": n0, "处理后": n1,
                "说明": (f"规则删除 {int(bad_sales.sum())} 条非正销售额、{int(bad_qty.sum())} 条非正数量、"
                        f"{int(bad_date.sum())} 条发货早于下单；IQR(3倍) 标记极端值 {int(df['is_outlier'].sum())} 条，保留不删")})
    return df.reset_index(drop=True)

def transform(df, log):
    df["Order Date"] = pd.to_datetime(df["Order Date"])
    df["Ship Date"] = pd.to_datetime(df["Ship Date"])
    for c in ["Sales", "Profit", "Discount"]:
        df[c] = df[c].astype(float).round(4)
    df["Quantity"] = df["Quantity"].astype(int)

    df["Order Year"] = df["Order Date"].dt.year
    df["Order Quarter"] = df["Order Date"].dt.quarter
    df["Order Month"] = df["Order Date"].dt.month
    df["Order YM"] = df["Order Date"].dt.to_period("M").astype(str)
    df["Order YQ"] = df["Order Date"].dt.to_period("Q").astype(str)

    df["Profit Margin"] = np.where(df["Sales"] > 0, df["Profit"] / df["Sales"], 0.0)
    df["Unit Price"] = df["Sales"] / df["Quantity"]
    df["Ship Days"] = (df["Ship Date"] - df["Order Date"]).dt.days
    df["Is Loss"] = df["Profit"] < 0
    df["Discount Band"] = pd.cut(df["Discount"], bins=DISC_BINS, labels=DISC_LABELS, include_lowest=True)
    df["Order Count Helper"] = 1

    log.append({"步骤": "类型与派生", "处理前": len(df), "处理后": len(df),
                "说明": "日期转 datetime；派生 Year/Quarter/Month/YM/YQ、利润率、单价、发货天数、亏损标记、折扣区间共 12 个衍生字段"})
    return df

def build_star(df):
    dim_customer = (df.sort_values("Order Date").groupby("Customer ID")
                      .agg(CustomerName=("Customer Name", "last"), Segment=("Segment", "last"),
                           City=("City", "last"), State=("State", "last"), Region=("Region", "last"),
                           PostalCode=("Postal Code", "last"), FirstOrder=("Order Date", "min"),
                           LastOrder=("Order Date", "max")).reset_index())
    dim_customer["TenureDays"] = (dim_customer["LastOrder"] - dim_customer["FirstOrder"]).dt.days

    dim_product = (df.groupby("Product ID")
                     .agg(ProductName=("Product Name", "last"), Category=("Category", "last"),
                          SubCategory=("Sub-Category", "last")).reset_index())

    dim_geo = (df.groupby(["State", "City"])
                 .agg(Region=("Region", "last"), PostalCode=("Postal Code", "last"))
                 .reset_index())

    dates = pd.date_range(df["Order Date"].min(), df["Order Date"].max(), freq="D")
    dim_date = pd.DataFrame({"Date": dates})
    dim_date["Year"] = dim_date["Date"].dt.year
    dim_date["Quarter"] = dim_date["Date"].dt.quarter
    dim_date["Month"] = dim_date["Date"].dt.month
    dim_date["MonthName"] = dim_date["Date"].dt.month_name()
    dim_date["DayOfWeek"] = dim_date["Date"].dt.dayofweek
    dim_date["IsWeekend"] = dim_date["DayOfWeek"].isin([5, 6])

    fact = df[["Row ID", "Order ID", "Order Date", "Ship Date", "Ship Mode", "Customer ID", 
               "Product ID", "State", "City", "Sales", "Quantity", "Discount", "Profit",
               "Profit Margin", "Unit Price", "Ship Days", "Is Loss", "Discount Band"]].copy()
    fact = fact.rename(columns={"Row ID": "RowID"})

    checks = {
        "fact.CustomerID ⊆ dim_customer": fact["Customer ID"].isin(dim_customer["Customer ID"]).all(),
        "fact.ProductID ⊆ dim_product": fact["Product ID"].isin(dim_product["Product ID"]).all(),
        "fact.(State,City) ⊆ dim_geo": fact.merge(dim_geo, on=["State", "City"], how="left")["Region"].notna().all(),
        "fact.OrderDate ⊆ dim_date": fact["Order Date"].isin(dim_date["Date"]).all(),
    }
    print("\n  外键校验：")
    for k, v in checks.items():
        print(f"    {k}: {'通过' if v else '失败'}")
    assert all(checks.values()), "主外键对齐失败"

    return {"fact_orders": fact, "dim_customer": dim_customer, 
            "dim_product": dim_product, "dim_geo": dim_geo, "dim_date": dim_date}
    
print("数据清洗")
raw = pd.read_csv(os.path.join(RAW_DIR, "orders_raw.csv"))
log = []

df = drop_duplicates(raw, log)
df = handle_missing(df, log)
df = handle_outliers(df, log)
df = transform(df, log)

print("\n  清洗日志：")
for r in log:
    print(f"    - {r['步骤']}：{r['处理前']} → {r['处理后']} 行｜{r['说明']}")

tables = build_star(df)
print("\n  星型模型表：")
for name, t in tables.items():
    print(f"    {name:14s} {t.shape[0]:>6,} 行 × {t.shape[1]:>2} 列")

for name, t in tables.items():
    t.to_csv(os.path.join(CLEAN_DIR, f"{name}.csv"), index=False, encoding="utf-8-sig")
tables["fact_orders"].to_json(os.path.join(CLEAN_DIR, "fact_orders.json"), orient="records", force_ascii=False, date_format="iso")
df.to_csv(os.path.join(CLEAN_DIR, "orders_clean.csv"), index=False, encoding="utf-8-sig")

db_path = os.path.join(CLEAN_DIR, "superstore.db")
with sqlite3.connect(db_path) as con:
    for name, t in tables.items():
        t.to_sql(name, con, index=False, if_exists="replace")
print(f"\n  导出 SQLite：superstore.db")

pd.DataFrame(log).to_csv(os.path.join(TAB_DIR, "t02_clean_log.csv"), index=False, encoding="utf-8-sig")
print("  已生成 t02_clean_log.csv")

fields = [
    ("Order ID", "订单编号", "字符串", "订单唯一标识，一个订单可含多个商品行"),
    ("Order Date", "下单日期", "日期", "2015-01-03 ~ 2018-12-30"),
    ("Ship Date", "发货日期", "日期", "用于计算发货时效"),
    ("Ship Mode", "配送方式", "类别", "四种：标准/二等/一等/当日"),
    ("Customer ID", "客户编号", "字符串", "外键 → dim_customer"),
    ("Segment", "客户细分", "类别", "Consumer / Corporate / Home Office"),
    ("Region", "销售大区", "类别", "West / East / Central / South"),
    ("State / City", "州 / 城市", "类别", "地理维度，外键 → dim_geo"),
    ("Product ID", "商品编号", "字符串", "外键 → dim_product"),
    ("Category", "商品大类", "类别", "Furniture / Office Supplies / Technology"),
    ("Sub-Category", "商品子类", "类别", "17 个，是波士顿矩阵与 ABC 分析的分析单元"),
    ("Sales", "销售额", "浮点", "美元（USD）原币口径，图表统一以「万美元」为单位"),
    ("Quantity", "销量", "整数", "件"),
    ("Discount", "折扣率", "浮点", "0 ~ 0.8，是利润侵蚀的主要解释变量"),
    ("Profit", "利润", "浮点", "可为负，负值即亏损行"),
]
pd.DataFrame(fields, columns=["字段", "中文名", "类型", "说明"]).to_csv(
    os.path.join(TAB_DIR, "t03_field_dict.csv"), index=False, encoding="utf-8-sig")
print("  已生成 t03_field_dict.csv")

数据清洗

  清洗日志：
    - 重复值处理：9994 → 9993 行｜Row ID 重复 0 条；业务键重复 1 条，均保留首条
    - 缺失值处理：11 → 0 行｜缺失字段 ['Postal Code']；按众数回填后被填 11 条
    - 异常值处理：9993 → 9993 行｜规则删除 0 条非正销售额、0 条非正数量、0 条发货早于下单；IQR(3倍) 标记极端值 1314 条，保留不删
    - 类型与派生：9993 → 9993 行｜日期转 datetime；派生 Year/Quarter/Month/YM/YQ、利润率、单价、发货天数、亏损标记、折扣区间共 12 个衍生字段

  外键校验：
    fact.CustomerID ⊆ dim_customer: 通过
    fact.ProductID ⊆ dim_product: 通过
    fact.(State,City) ⊆ dim_geo: 通过
    fact.OrderDate ⊆ dim_date: 通过

  星型模型表：
    fact_orders     9,993 行 × 18 列
    dim_customer      793 行 × 10 列
    dim_product     1,862 行 ×  4 列
    dim_geo           604 行 ×  4 列
    dim_date        1,458 行 ×  7 列

  导出 SQLite：superstore.db
  已生成 t02_clean_log.csv
  已生成 t03_field_dict.csv
